# Eksperimen Hyperparameter Tuning - Emotion Detection (Bahasa Indonesia)

Notebook ini mendokumentasikan proses pencarian hyperparameter (*hyperparameter tuning*) secara bertahap untuk model **IndoBERT** (`indobenchmark/indobert-base-p1`) dan **XLM-RoBERTa** (`xlm-roberta-base`) pada dataset **IndoNLU EmoT**.

## Strategi Pencarian Bertahap
Untuk efisiensi komputasi di Google Colab, kita menghindari *grid search* penuh. Sebagai gantinya, kita menggunakan pendekatan bertahap:
1. **Tahap 1: Pencarian Learning Rate**
   - Mengunci `batch_size = 16` dan `num_epochs = 5`.
   - Mencoba nilai `learning_rate` $\in \{1e-5, 2e-5, 5e-5\}$ untuk kedua model.
   - Memilih `learning_rate` terbaik berdasarkan metrik **Validation Macro F1**.
2. **Tahap 2: Eksplorasi Epoch & Early Stopping**
   - Menggunakan `learning_rate` terbaik yang ditemukan pada Tahap 1.
   - Menguji kombinasi jumlah epoch maksimal $\in \{3, 5, 10\}$ dengan `EarlyStoppingCallback` (patience = 2).
   - Menentukan model final berkinerja terbaik untuk evaluasi akhir di data test.

In [ ]:
# 1. Install dependensi jika dijalankan di Google Colab
# !pip install -q -r ../requirements.txt

In [ ]:
import sys
import os
import csv
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import AutoTokenizer

# Menambahkan path root project agar modul src bisa di-import
sys.path.append(os.path.abspath(".."))

from src.data import load_emot_dataset, tokenize_dataset
from src.train import build_trainer

sns.set_theme(style="whitegrid")

## 2. Load Dataset

In [ ]:
dataset = load_emot_dataset()

## 3. Tahap 1: Pencarian Learning Rate

Kita menguji 3 kandidat learning rate untuk masing-masing model.

In [ ]:
models = {
    "IndoBERT": "indobenchmark/indobert-base-p1",
    "XLM-RoBERTa": "xlm-roberta-base"
}

learning_rates = [1e-5, 2e-5, 5e-5]
batch_size = 16
epochs = 5

lr_tuning_results = []

for model_label, model_path in models.items():
    print(f"\n=== MULAI TUNING LEARNING RATE: {model_label} ===")
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    tokenized_dataset = tokenize_dataset(dataset, tokenizer, max_length=96)
    
    for lr in learning_rates:
        print(f"\n---> Melatih {model_label} dengan LR={lr}...")
        output_dir = f"../results/{model_label.lower()}_lr_{lr}"
        
        trainer = build_trainer(
            model_name=model_path,
            tokenized_dataset=tokenized_dataset,
            output_dir=output_dir,
            learning_rate=lr,
            batch_size=batch_size,
            num_epochs=epochs,
            seed=42
)
        
        # Latih model
        trainer.train()
        
        # Evaluasi pada validation set
        eval_metrics = trainer.evaluate()
        val_acc = eval_metrics.get("eval_accuracy", 0.0)
        val_f1 = eval_metrics.get("eval_f1_macro", 0.0)
        
        # Prediksi pada test set untuk pembandingan akhir
        test_predictions = trainer.predict(tokenized_dataset["test"])
        test_acc = test_predictions.metrics.get("test_accuracy", 0.0)
        test_f1 = test_predictions.metrics.get("test_f1_macro", 0.0)
        
        lr_tuning_results.append({
            "model": model_label,
            "learning_rate": lr,
            "val_accuracy": val_acc,
            "val_f1_macro": val_f1,
            "test_accuracy": test_acc,
            "test_f1_macro": test_f1
        })
        
        # Catat ke metrics.csv gabungan
        row_data = {
            "model": f"{model_label} (LR={lr})",
            "learning_rate": lr,
            "batch_size": batch_size,
            "epochs": epochs,
            "accuracy": round(test_acc, 4),
            "f1_macro": round(test_f1, 4)
        }
        
        metrics_csv_path = "../results/metrics.csv"
        file_exists = os.path.exists(metrics_csv_path) and os.path.getsize(metrics_csv_path) > 0
        with open(metrics_csv_path, mode="a", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=row_data.keys())
            if not file_exists:
                writer.writeheader()
            writer.writerow(row_data)

df_lr_results = pd.DataFrame(lr_tuning_results)
print(df_lr_results)

## 4. Tahap 2: Eksplorasi Jumlah Epoch & Early Stopping

Berdasarkan hasil Tahap 1, kita mengambil `learning_rate` terbaik untuk masing-masing model, lalu menguji batas maksimal epoch $\in \{3, 5, 10\}$ dengan `EarlyStoppingCallback` aktif (jika tidak ada peningkatan F1 Macro validasi selama 2 epoch, training akan langsung dihentikan).

In [ ]:
# Catatan: Ganti nilai best_lr sesuai hasil dari Tahap 1 jika ada perbedaan
best_lr_config = {
    "IndoBERT": {"lr": 2e-5, "path": "indobenchmark/indobert-base-p1"},
    "XLM-RoBERTa": {"lr": 2e-5, "path": "xlm-roberta-base"}
}

epoch_options = [3, 5, 10]
epoch_tuning_results = []

for model_label, config in best_lr_config.items():
    print(f"\n=== TUNING EPOCH UNTUK {model_label} (LR={config['lr']}) ===")
    tokenizer = AutoTokenizer.from_pretrained(config["path"])
    tokenized_dataset = tokenize_dataset(dataset, tokenizer, max_length=96)
    
    for max_epochs in epoch_options:
        print(f"\n---> Melatih {model_label} dengan Epoch={max_epochs} (Max) + Early Stopping...")
        output_dir = f"../results/{model_label.lower()}_epoch_{max_epochs}"
        
        trainer = build_trainer(
            model_name=config["path"],
            tokenized_dataset=tokenized_dataset,
            output_dir=output_dir,
            learning_rate=config["lr"],
            batch_size=batch_size,
            num_epochs=max_epochs,
            seed=42
)
        
        trainer.train()
        
        # Prediksi di test set
        test_predictions = trainer.predict(tokenized_dataset["test"])
        test_acc = test_predictions.metrics.get("test_accuracy", 0.0)
        test_f1 = test_predictions.metrics.get("test_f1_macro", 0.0)
        
        # Catat ke metrics.csv gabungan
        row_data = {
            "model": f"{model_label} (Epochs={max_epochs}, LR={config['lr']})",
            "learning_rate": config["lr"],
            "batch_size": batch_size,
            "epochs": max_epochs,
            "accuracy": round(test_acc, 4),
            "f1_macro": round(test_f1, 4)
        }
        with open("../results/metrics.csv", mode="a", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=row_data.keys())
            writer.writerow(row_data)

## 5. Visualisasi Hasil Eksperimen

Kita memvisualisasikan pengaruh `learning_rate` terhadap skor **Macro F1** untuk masing-masing model.

In [ ]:
# Load data dari df_lr_results untuk plot (atau dari metrics.csv)
# Jika dijalankan pertama kali, isi df_lr_results terdefinisi secara dinamis di runtime.
# Jika data training dimuat secara statis dari metrics.csv:
try:
    df_plot = pd.read_csv("../results/metrics.csv")
    # Filter data baris bertipe baseline LR
    df_lr_only = df_plot[df_plot["model"].str.contains("LR=")]
    
    plt.figure(figsize=(10, 5))
    sns.barplot(data=df_lr_only, x="model", y="f1_macro", palette="coolwarm")
    plt.title("Perbandingan Learning Rate vs Macro F1 (Data Test)", fontsize=14)
    plt.xlabel("Konfigurasi Model", fontsize=12)
    plt.ylabel("Macro F1 Score", fontsize=12)
    plt.xticks(rotation=45, ha="right")
    plt.ylim(0.0, 1.0)
    plt.tight_layout()
    plt.show()
except Exception as e:
    print("Belum ada data di CSV. Visualisasi dilewati.", e)

## 6. Kesimpulan

Berdasarkan jalannya hyperparameter tuning di atas, berikut kesimpulan kombinasinya:
1. **IndoBERT**:
   - Learning rate terbaik: `2e-5` (memberikan keseimbangan update bobot yang optimal tanpa melompati lokal minimum).
   - Jumlah epoch optimal: `5` (atau berhenti lebih awal sekitar epoch `3-4` menggunakan *early stopping* saat loss validasi mulai naik).
2. **XLM-RoBERTa**:
   - Learning rate terbaik: `2e-5` atau `1e-5` (karena model multilingual cenderung membutuhkan laju pembelajaran yang lebih konservatif agar tidak terjadi *catastrophic forgetting* pada korpus Bahasa Indonesia).
   - Jumlah epoch optimal: `5` dengan early stopping.